# Data Cleaning and Transformation
Project : Azure E-Commerce data Pipeline

Done by : Saqib Souran


In [0]:
# Configuration variables
storage_account = "MyADLSStorage"
application_id = "dadadq34-dad34d-dd3-fas3-fgsdkfu45464"
directory_id = "brthrtg-brtbrttrg-brtbrtbrt-bhrtr-11hrthrtgrt"
service_credential_key = "dugb8~gieuar32eryfgghgegersg4423gjk4g2b42g4"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", application_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", service_credential_key)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{directory_id}/oauth2/token")

In [0]:

# Read CSV file from ADLS Gen2 using Spark DataFrame
storage_account = "MyADLSStorage"
container_name = "olistdata"
adls_bronze_path =  f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/bronze/"
adls_silver_path =  f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/silver/"
customer_path = adls_bronze_path + "olist_customers_dataset.csv"
geolocation_path = adls_bronze_path + "olist_geolocation_dataset.csv"
items_path = adls_bronze_path + "olist_order_items_dataset.csv"
payments_path = adls_bronze_path + "olist_order_payments_dataset"
reviews_path = adls_bronze_path + "olist_order_reviews_dataset.csv"
sellers_path = adls_bronze_path + "olist_sellers_dataset.csv"
products_path = adls_bronze_path + "olist_products_dataset.csv"
orders_path = adls_bronze_path + "olist_orders_dataset.csv"

# Read CSV file from ADLS Gen
customers_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(customer_path)
geolocation_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(geolocation_path)
items_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(items_path)
payments_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(payments_path)
reviews_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(reviews_path)
sellers_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(sellers_path)
products_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(products_path)
orders_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(orders_path)



In [0]:
customers_df.show(10)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
|879864dab9bc30475...|4c93744516667ad3b...|                   89254|      jaragua do sul|            SC|
|fd826e7cf63160e53...|addec96d2e059c80c...|            

In [0]:
# Identify Missing Values
from pyspark.sql.functions import *
def missing_values(df,df_name):
    print(f'Missing values in {df_name}:')
    df.select([count(when(col(c).isNull(),1)).alias(c) for c in df.columns]).show()


In [0]:
missing_values(customers_df,'customer')

Missing values in customer:
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



In [0]:
missing_values(orders_df,'orders')

Missing values in orders:
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



In [0]:
missing_values(items_df,'order_item')

Missing values in order_item:
+--------+-------------+----------+---------+-------------------+-----+-------------+
|order_id|order_item_id|product_id|seller_id|shipping_limit_date|price|freight_value|
+--------+-------------+----------+---------+-------------------+-----+-------------+
|       0|            0|         0|        0|                  0|    0|            0|
+--------+-------------+----------+---------+-------------------+-----+-------------+



In [0]:
missing_values(payments_df,'payments')

Missing values in payments:
+--------+------------------+------------+--------------------+-------------+
|order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------+------------------+------------+--------------------+-------------+
|       0|                 0|           0|                   0|            0|
+--------+------------------+------------+--------------------+-------------+



# Handle Missing Values

1. Drop missing Values ( for non - critical columns )

2. Fill missing values ( for numerical columns )

3. Impute Missing Values ( for continous data )

In [0]:
orders_df_cleaned = orders_df.na.drop(subset=['order_id','customer_id','order_status'])

In [0]:
orders_df_cleaned.show(10)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [0]:
orders_df_cleaned = orders_df.fillna({'order_delivered_customer_date':'9999-12-31'})

In [0]:
orders_df_cleaned.show(10)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

# Impute missing values 

In [0]:
# Calculate the mean, ignoring null values
mean_value = payments_df.select(mean("payment_value")).first()[0]

# Create an imputed column
payments_df_cleaned = payments_df.withColumn("payment_value_imputed",coalesce(col("payment_value"), lit(mean_value)))

payments_df_cleaned.show(10)

+--------------------+------------------+------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+------------------+------------+--------------------+-------------+---------------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|   99.3300018|           99.3300018|
|a9810da82917af2d9...|                 1| credit_card|                   1|   24.3899994|           24.3899994|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|   65.7099991|           65.7099991|
|ba78997921bbcdc13...|                 1| credit_card|                   8|   107.779999|           107.779999|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|   128.449997|           128.449997|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|   96.1200027|           96.1

In [0]:
payments_df_with_null = payments_df.withColumn('payment_value',when(col('payment_value')!= 99.33,col('payment_value')).otherwise(lit(None)))

In [0]:
payments_df_with_null.show(10)

+--------------------+------------------+------------+--------------------+-------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|
+--------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|   99.3300018|
|a9810da82917af2d9...|                 1| credit_card|                   1|   24.3899994|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|   65.7099991|
|ba78997921bbcdc13...|                 1| credit_card|                   8|   107.779999|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|   128.449997|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|   96.1200027|
|771ee386b001f0620...|                 1| credit_card|                   1|   81.1600037|
|3d7239c394a212faa...|                 1| credit_card|                   3|   51.8400002|
|1f78449c8

In [0]:
# Calculate the median, ignoring null values
median_value = payments_df_with_null.approxQuantile("payment_value", [0.5], 0.001)[0]

# Replace null values with the median
payments_df_cleaned = payments_df_with_null.withColumn("payment_value_imputed",coalesce(col("payment_value"),lit(median_value)))

payments_df_cleaned.show(10)

+--------------------+------------------+------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+------------------+------------+--------------------+-------------+---------------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|   99.3300018|           99.3300018|
|a9810da82917af2d9...|                 1| credit_card|                   1|   24.3899994|           24.3899994|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|   65.7099991|           65.7099991|
|ba78997921bbcdc13...|                 1| credit_card|                   8|   107.779999|           107.779999|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|   128.449997|           128.449997|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|   96.1200027|           96.1

##Standardizing the Format

In [0]:
def print_schema(df,df_name):
    print(f'schema of {df_name}:')
    df.printSchema()

In [0]:
print_schema(orders_df,'orders')

schema of orders:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



In [0]:
print_schema(customers_df,'customers')

schema of customers:
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [0]:
print_schema(payments_df,'payments')

schema of payments:
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



In [0]:
orders_df.show(10)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [0]:
orders_df_cleaned = orders_df_cleaned.withColumn('order_purchase_timestamp',to_date(col('order_purchase_timestamp')))

In [0]:
orders_df_cleaned.show(10)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|              2017-10-02|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|              2018-07-24|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [0]:
payments_df_cleaned.show(10)

+--------------------+------------------+------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential|payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+------------------+------------+--------------------+-------------+---------------------+
|b81ef226f3fe1789b...|                 1| credit_card|                   8|   99.3300018|           99.3300018|
|a9810da82917af2d9...|                 1| credit_card|                   1|   24.3899994|           24.3899994|
|25e8ea4e93396b6fa...|                 1| credit_card|                   1|   65.7099991|           65.7099991|
|ba78997921bbcdc13...|                 1| credit_card|                   8|   107.779999|           107.779999|
|42fdf880ba16b47b5...|                 1| credit_card|                   2|   128.449997|           128.449997|
|298fcdf1f73eb413e...|                 1| credit_card|                   2|   96.1200027|           96.1

In [0]:
payments_df_cleaned = payments_df_cleaned.withColumn('payment_type',when(col('payment_type')=='boleto','Bank Transfer')
                                                     .when(col('payment_type')=='credit_card','Credit Card')
                                                     .when(col('payment_type')=='debit_card','Debit Card')
                                                    .otherwise('other'))

In [0]:
payments_df_cleaned.show(10)

+--------------------+------------------+-------------+--------------------+-------------+---------------------+
|            order_id|payment_sequential| payment_type|payment_installments|payment_value|payment_value_imputed|
+--------------------+------------------+-------------+--------------------+-------------+---------------------+
|b81ef226f3fe1789b...|                 1|  Credit Card|                   8|   99.3300018|           99.3300018|
|a9810da82917af2d9...|                 1|  Credit Card|                   1|   24.3899994|           24.3899994|
|25e8ea4e93396b6fa...|                 1|  Credit Card|                   1|   65.7099991|           65.7099991|
|ba78997921bbcdc13...|                 1|  Credit Card|                   8|   107.779999|           107.779999|
|42fdf880ba16b47b5...|                 1|  Credit Card|                   2|   128.449997|           128.449997|
|298fcdf1f73eb413e...|                 1|  Credit Card|                   2|   96.1200027|      

In [0]:
customers_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [0]:
customers_df.show(10)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
|879864dab9bc30475...|4c93744516667ad3b...|                   89254|      jaragua do sul|            SC|
|fd826e7cf63160e53...|addec96d2e059c80c...|            

In [0]:
customers_df_cleaned = customers_df.withColumn('customer_zip_code_prefix',col('customer_zip_code_prefix').cast('string'))

In [0]:
customers_df_cleaned.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



### Remove Duplicate Records

In [0]:
customers_df_cleaned = customers_df_cleaned.dropDuplicates(['customer_id'])

In [0]:
order_with_details = orders_df_cleaned.join(items_df,'order_id','left')\
.join(payments_df_cleaned,'order_id','left')\
.join(customers_df_cleaned,'customer_id','left')

In [0]:
order_with_details.show(10)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+--------------------+--------------------+-------------------+------+-------------+------------------+-------------+--------------------+-------------+---------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|            order_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|payment_sequential| payment_type|payment_installments|payment_value|payment_value_imputed|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------+------------------------

In [0]:
order_with_total_value = order_with_details.groupBy('order_id')\
.agg(sum('payment_value').alias('total_order_value'))

In [0]:
order_with_total_value.show(5)

+--------------------+------------------+
|            order_id| total_order_value|
+--------------------+------------------+
|f373335aac9a659de...|        59.1800003|
|118045506e1c1dda0...|1801.9999699999998|
|cc66dee6fbc18bb79...|        136.399994|
|f44cb69655f8e4d13...|        164.320007|
|edcc6b79e8394346b...|        162.630005|
+--------------------+------------------+
only showing top 5 rows


In [0]:
# Average Delivery Time Analysis
orders_df_cleaned = orders_df_cleaned.withColumn('delivery_time',datediff(col('order_delivered_customer_date'),col('order_purchase_timestamp')))

In [0]:
orders_df_cleaned.select('order_id','order_purchase_timestamp','order_delivered_customer_date','delivery_time').orderBy('delivery_time',ascending=False).show(10)


+--------------------+------------------------+-----------------------------+-------------+
|            order_id|order_purchase_timestamp|order_delivered_customer_date|delivery_time|
+--------------------+------------------------+-----------------------------+-------------+
|ca07593549f1816d2...|              2017-02-21|          2017-09-19 14:36:39|          210|
|1b3190b2dfa9d789e...|              2018-02-23|          2018-09-19 23:24:07|          208|
|440d0d17af552815d...|              2017-03-07|          2017-09-19 15:12:50|          196|
|2fb597c2f772eca01...|              2017-03-08|          2017-09-19 14:33:17|          195|
|285ab9426d6982034...|              2017-03-08|          2017-09-19 14:00:04|          195|
|0f4519c5f1c541dde...|              2017-03-09|          2017-09-19 14:38:21|          194|
|47b40429ed8cce3ae...|              2018-01-03|          2018-07-13 20:51:31|          191|
|2fe324febf907e3ea...|              2017-03-13|          2017-09-19 17:00:07|   

## Advance Transformation

In [0]:
items_df.show(10)

+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date| price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+------+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35|  58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13| 239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30| 199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18| 12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51| 199.9|        18.14|
|00048cc3ae777c65d...|            1|ef92

In [0]:
quantiles = items_df.approxQuantile('price',[0.01,0.99],0.0)
low_cutoff,high_cutoff = quantiles[0],quantiles[1]

In [0]:
items_df.select('price').summary().show()

+-------+-----------------+
|summary|            price|
+-------+-----------------+
|  count|           112650|
|   mean|120.6537390146417|
| stddev|183.6339280502595|
|    min|             0.85|
|    25%|             39.9|
|    50%|            74.99|
|    75%|            134.9|
|    max|           6735.0|
+-------+-----------------+



In [0]:
low_cutoff,high_cutoff

(9.99, 890.0)

In [0]:
order_item_df_cleaned = items_df.filter((col('price') >=low_cutoff) & (col('price') <=high_cutoff))

In [0]:
payments_df_cleaned.select('payment_installments').summary().show()

+-------+--------------------+
|summary|payment_installments|
+-------+--------------------+
|  count|              103886|
|   mean|   2.853348863176944|
| stddev|  2.6870506738564797|
|    min|                   0|
|    25%|                   1|
|    50%|                   1|
|    75%|                   4|
|    max|                  24|
+-------+--------------------+



In [0]:
products_df.show(10)

+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|          product_id|product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|1e9e8ef04dbcff454...|           perfumaria|                 40|                       287|                 1|             225|               16|               10|              14|
|3aa071139cb16b67c...|                artes|                 44|                       276|                 1|            1000|               30|               18|              20|
|96bd76ec8810374ed...|        esporte_lazer|                 46|                       250|    

In [0]:
products_df_cleaned = products_df.withColumn(
    'product_size_category',
    when(col('product_weight_g') <500,'Small')
    .when(col('product_weight_g').between(500,2000),'Medium')
    .otherwise('Large')
)

#### Connecting to MongoDB database online for Data Enrichment (product name translation)

**Context:**

We are doing data enrichment on Product table. The product names are written in another language and we have a raw data on our MongoDB database where we have the name of these products in English.

** What we will do**

- Connect to MongoDb database online,
- Import the Translation table
- Add the English Names as well to Product dataframe

<em>username and password removed</em>

In [0]:
# importing module
from pymongo import MongoClient

hostname = "6znl6a.h.filess.io"
database = "olistDB_Nosql_sheeasily"
port = "27018"
username = "UUUUUUUUUUU"
password = "DDDDDDDDDDDDDDDDDDD"

uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + port + "/" + database

# Connect with the portnumber and host
client = MongoClient(uri)

# Access database
mydatabase = client[database]
print(mydatabase.name)




olistDB_Nosql_sheeasily


In [0]:
import pandas as pd
collection = mydatabase['product_category_name_translation']

mongo_data = pd.DataFrame(list(collection.find()))
mongo_data.head()

,_id,product_category_name,product_category_name_english
0,6ad7b8551e572c7d0f4080a0,beleza_saude,health_beauty
1,6ad7b8551e572c7d0f4080a1,informatica_acessorios,computers_accessories
2,6ad7b8551e572c7d0f4080a2,automotivo,auto
3,6ad7b8551e572c7d0f4080a3,cama_mesa_banho,bed_bath_table
4,6ad7b8551e572c7d0f4080a4,moveis_decoracao,furniture_decor


In [0]:
mongo_data.drop('_id', axis=1, inplace=True)
mongo_spark_df = spark.createDataFrame(mongo_data)

mongo_spark_df.show(5)

+---------------------+-----------------------------+
|product_category_name|product_category_name_english|
+---------------------+-----------------------------+
|         beleza_saude|                health_beauty|
| informatica_acess...|         computers_accesso...|
|           automotivo|                         auto|
|      cama_mesa_banho|               bed_bath_table|
|     moveis_decoracao|              furniture_decor|
+---------------------+-----------------------------+
only showing top 5 rows


In [0]:
products_df_cleaned = products_df_cleaned.join(mongo_spark_df, "product_category_name", "left")

In [0]:
products_df_cleaned.show(10)

+---------------------+--------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+---------------------+-----------------------------+
|product_category_name|          product_id|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|product_size_category|product_category_name_english|
+---------------------+--------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+---------------------+-----------------------------+
|           perfumaria|1e9e8ef04dbcff454...|                 40|                       287|                 1|             225|               16|               10|              14|                Small|                    perfumery|
|                artes|3aa071139cb16b67c...|                 44|    

In [0]:
# Calculate total revenue per seller
total_revenue_per_seller = items_df.groupBy('seller_id') \
    .agg(round(sum(col('price') + col('freight_value')), 2).alias('total_revenue')) \
    .orderBy('total_revenue', ascending=False)

total_revenue_per_seller.show(10)

+--------------------+-------------+
|           seller_id|total_revenue|
+--------------------+-------------+
|4869f7a5dfa277a7d...|     249640.7|
|7c67e1448b00f6e96...|    239536.44|
|53243585a1d6dc264...|    235856.68|
|4a3ca9315b744ce9f...|    235539.96|
|fa1c13f2614d7b5c4...|    204084.73|
|da8622b14eb17ae28...|    185192.32|
|7e93a43ef30c4f03f...|    182754.05|
|1025f0e2d44d7041d...|    172860.69|
|7a67c85e85bb2ce85...|    162648.38|
|955fee9216a65b617...|    160602.68|
+--------------------+-------------+
only showing top 10 rows


### Geolocation Dataset Cleaning

In [0]:
geolocation_df.show(10)

+---------------------------+-------------------+------------------+----------------+-----------------+
|geolocation_zip_code_prefix|    geolocation_lat|   geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+-------------------+------------------+----------------+-----------------+
|                       1037| -23.54562128115268|-46.63929204800168|       sao paulo|               SP|
|                       1046|-23.546081127035535|-46.64482029837157|       sao paulo|               SP|
|                       1046| -23.54612896641469|-46.64295148361138|       sao paulo|               SP|
|                       1041|  -23.5443921648681|-46.63949930627844|       sao paulo|               SP|
|                       1035|-23.541577961711493|-46.64160722329613|       sao paulo|               SP|
|                       1012|-23.547762303364266|-46.63536053788448|       são paulo|               SP|
|                       1047|-23.546273112412678|-46.64122516971

In [0]:
geolocation_df.printSchema()

root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)



In [0]:
geolocation_df = geolocation_df.select(
    trim(col("geolocation_zip_code_prefix")).alias("geolocation_zip_code_prefix"),
    col("geolocation_lat"),
    col("geolocation_lng"),
    initcap(trim(col("geolocation_city"))).alias("geolocation_city"),
    upper(trim(col("geolocation_state"))).alias("geolocation_state")
    )

geolocation_df.show(10)

+---------------------------+-------------------+------------------+----------------+-----------------+
|geolocation_zip_code_prefix|    geolocation_lat|   geolocation_lng|geolocation_city|geolocation_state|
+---------------------------+-------------------+------------------+----------------+-----------------+
|                       1037| -23.54562128115268|-46.63929204800168|       Sao Paulo|               SP|
|                       1046|-23.546081127035535|-46.64482029837157|       Sao Paulo|               SP|
|                       1046| -23.54612896641469|-46.64295148361138|       Sao Paulo|               SP|
|                       1041|  -23.5443921648681|-46.63949930627844|       Sao Paulo|               SP|
|                       1035|-23.541577961711493|-46.64160722329613|       Sao Paulo|               SP|
|                       1012|-23.547762303364266|-46.63536053788448|       São Paulo|               SP|
|                       1047|-23.546273112412678|-46.64122516971

In [0]:
geolocation_df = geolocation_df.filter(
    col("geolocation_zip_code_prefix").rlike("^[0-9]{5}$")
    & col("geolocation_lat").between(-34, 6)
    & col("geolocation_lng").between(-74, -28)
    & col("geolocation_city").isNotNull()
    & (col("geolocation_city") != "")
    & col("geolocation_state").rlike("^[A-Z]{2}$")
)


In [0]:
geolocation_df.select("geolocation_city").distinct().show(10)

+----------------+
|geolocation_city|
+----------------+
|          Santos|
|         Guarujá|
|        Ilhabela|
|        Maresias|
|         Cubatao|
|   Caraguatatuba|
|         Cubatão|
|         Guaruja|
|     São Vicente|
|         Ubatuba|
+----------------+
only showing top 10 rows


In [0]:
accents =   "áàâäãåçéèêëíìîïñóòôöõúùûüýÿÁÀÂÄÃÅÇÉÈÊËÍÌÎÏÑÓÒÔÖÕÚÙÛÜÝ"
unaccented = "aaaaaaceeeeiiiinooooouuuuyyAAAAAACEEEEIIIINOOOOOUUUUY"

geolocation_df = geolocation_df.withColumn(
    "city_clean", 
    translate(col("geolocation_city"), accents, unaccented)
)

In [0]:
geolocation_df = geolocation_df.drop("geolocation_city")
geolocation_df = geolocation_df.withColumnRenamed("city_clean", "geolocation_city")
geolocation_df_cleaned = geolocation_df.dropDuplicates()

In [0]:
geolocation_df_cleaned.show(10)

+---------------------------+-------------------+-------------------+-----------------+----------------+
|geolocation_zip_code_prefix|    geolocation_lat|    geolocation_lng|geolocation_state|geolocation_city|
+---------------------------+-------------------+-------------------+-----------------+----------------+
|                      11030|-23.981964066882146|-46.310039559264744|               SP|          Santos|
|                      11065|-23.966295653067107| -46.34720400721336|               SP|          Santos|
|                      11045|-23.972011770262974| -46.32430590750893|               SP|          Santos|
|                      11045| -23.96112504302036| -46.32316061267247|               SP|          Santos|
|                      11065|  -23.9678895545329| -46.34412123980033|               SP|          Santos|
|                      11060|-23.967052416613658| -46.33691592783774|               SP|          Santos|
|                      11013|-23.933071987144555| -46.3

### Reviews Dataset Cleaning

In [0]:
reviews_df.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)



In [0]:
# cleaning and transformation
reviews_df = (
    reviews_df.withColumn("review_score", col("review_score").cast("int")) \
    .withColumn("review_creation_date", to_timestamp("review_creation_date"))\
    .withColumn("review_answer_timestamp", to_timestamp("review_answer_timestamp"))\
    .withColumn("review_comment_title", trim(col("review_comment_title")))\
    .withColumn("review_comment_message", trim(col("review_comment_message")))\
    .dropDuplicates()
)

In [0]:
# Keep valid reviews
reviews_df = reviews_df.filter(
    col("review_id").isNotNull()
    & col("order_id").isNotNull()
    & col("review_score").between(1, 5)
)

In [0]:
# Improving Data Quality
reviews_df_cleaned = reviews_df.withColumn(
    "review_sentiment", when(col("review_score") >= 4, "Positive")\
     .when(col("review_score") == 3, "Neutral")\
     .otherwise("Negative")
)

In [0]:
reviews_df_cleaned.show()

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+----------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|review_sentiment|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+----------------+
|94c4db81ff5c234f5...|3fb155b4897652f5c...|           5|                NULL|                  NULL| 2017-12-15 00:00:00|    2017-12-17 14:43:35|        Positive|
|3a9cc3b90bc22af3b...|8071505187465938d...|           1|                NULL|  Boa noite , não g...|                NULL|                   NULL|        Negative|
|44bb8c2ad3bb9c2bb...|56c79c2be1ceff583...|           2|           Bebedouro|  Bom dia ... compr...| 2018-05-08 00:00:00|    2018-05-11 11:03:01|        Negative|
|d326ebc1ded6003be...|

###Sellers Dataset Cleaning

In [0]:
sellers_df.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



In [0]:
# Remove duplicate rows
sellers_df = sellers_df.dropDuplicates()

In [0]:
# Trim whitespace from string columns
for col_name in sellers_df.columns:
    sellers_df = sellers_df.withColumn(
        col_name,
        trim(col(col_name).cast("string"))
    )


In [0]:
# Convert empty strings to NULL
for col_name in sellers_df.columns:
    sellers_df = sellers_df.withColumn(
        col_name,
        when(col(col_name) == "", None)
         .otherwise(col(col_name))
    )

In [0]:
# Remove records with missing seller IDs
sellers_df = sellers_df.filter(
    col("seller_id").isNotNull()
)


In [0]:
# Standardize seller city and state
sellers_df = (sellers_df
    .withColumn(
        "seller_city",
        initcap(lower(col("seller_city")))
    )
    .withColumn(
        "seller_state",
        upper(col("seller_state"))
    )
)

In [0]:
# Convert ZIP code to string
sellers_df = sellers_df.withColumn(
    "seller_zip_code_prefix",
    lpad(col("seller_zip_code_prefix").cast("string"), 5, "0")
)

In [0]:
# Remove duplicate seller IDs
sellers_df_cleaned = sellers_df.dropDuplicates(["seller_id"])

In [0]:
# So here we the list of all cleaned DataFrames to save as Parquet files in our silver Layer

# 1. Cleaned Orders DataFrame
orders_df_cleaned.write.mode('overwrite').parquet(adls_silver_path + 'orders_cleaned/')

# 2. Cleaned Items DataFrame
order_item_df_cleaned.write.mode('overwrite').parquet(adls_silver_path + 'items_cleaned/')

# 3. Cleaned Payments DataFrame
payments_df_cleaned.write.mode('overwrite').parquet(adls_silver_path + 'payments_cleaned/')

# 4. Cleaned Customers DataFrame
customers_df_cleaned.write.mode('overwrite').parquet(adls_silver_path + 'customers_cleaned/')

# 5. Cleaned Products DataFrame (with product_size_category)
products_df_cleaned.write.mode('overwrite').parquet(adls_silver_path + 'products_cleaned/')

# 6. Order with Details (joined DataFrame)
order_with_details.write.mode('overwrite').parquet(adls_silver_path + 'order_with_details/')

# 7. Order with Total Value (aggregated DataFrame)
order_with_total_value.write.mode('overwrite').parquet(adls_silver_path + 'order_with_total_value/')

# 8. Total Revenue Per Seller
total_revenue_per_seller.write.mode('overwrite').parquet(adls_silver_path + 'total_revenue_per_seller/')

# 9. Cleaned Geolocation Daraframe
geolocation_df_cleaned.write.mode('overwrite').parquet(adls_silver_path + 'geolocation_cleaned/')

# 10. Cleaned Reviews Daraframe
reviews_df_cleaned.write.mode('overwrite').parquet(adls_silver_path + 'reviews_cleaned/')

# 11. Cleaned Sellers Daraframe
sellers_df_cleaned.write.mode('overwrite').parquet(adls_silver_path + 'sellers_cleaned/')

print("All cleaned DataFrames saved as Parquet files successfully!")

All cleaned DataFrames saved as Parquet files successfully!
